# Notebook 1: Feature Extraction from Tree Crown Polygons

This notebook extracts morphological features from crown elevation polygons for predicting tree trunk count. Two feature sets are provided:

1. **5-Feature Set** (original model): perimeter, area, compactness, perimeter-to-area ratio, eccentricity
2. **20-Feature Set** (expanded): adds convexity, MRR axes, boundary complexity, radial stats, and derived metrics

The output is a feature-enriched shapefile (or GeoDataFrame) ready for model training in Notebook 2.

## 1.1 Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import math
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
SHAPEFILE_PATH = "train_set_validated.shp"  # Path to input shapefile
TARGET_CRS = 2039  # EPSG:2039 (Israel Transverse Mercator, units in meters)

# Load and prepare data
gdf = gpd.read_file(SHAPEFILE_PATH)
gdf = gdf.to_crs(epsg=TARGET_CRS)
gdf = gdf.explode(index_parts=False).reset_index(drop=True)
gdf = gdf[gdf.geometry.type == 'Polygon'].copy()

print(f"Loaded {len(gdf)} polygons")
print(f"CRS: {gdf.crs}")
print(f"Columns: {list(gdf.columns)}")
gdf.head()

## 1.2 Remove Contained Polygons

Some polygons may be fully contained within larger ones (artifacts from crown delineation). These are removed using a spatial index for efficiency.

In [ ]:
n_before = len(gdf)
sindex = gdf.sindex
to_drop = set()
for idx, row in gdf.iterrows():
    if idx in to_drop:
        continue
    candidates = list(sindex.query(row.geometry, predicate='contains'))
    for c in candidates:
        if c != idx and c not in to_drop:
            if row.geometry.contains(gdf.iloc[c].geometry):
                to_drop.add(c)

gdf = gdf.drop(index=list(to_drop)).reset_index(drop=True)
print(f"Removed {n_before - len(gdf)} contained polygons. Remaining: {len(gdf)}")

## 1.3 Option A: 5-Feature Set (Original Model)

The original model uses 5 features derived from basic polygon geometry:

| Feature | Formula | Description |
|---------|---------|-------------|
| perimeter | `geometry.length` | Boundary length in meters |
| area | `geometry.area` | Polygon area in sq meters |
| compactness | `4 * pi * area / perimeter^2` | Isoperimetric quotient (1 = circle) |
| perimeter_to_area | `perimeter / area` | Shape complexity ratio |
| eccentricity | `sqrt(1 - minor^2/major^2)` | Elongation (0 = circle, ~1 = line) |

Eccentricity is derived from the minimum rotated rectangle (MRR) axis lengths.

In [ ]:
def extract_5_features(gdf):
    """Extract the original 5-feature set from a GeoDataFrame of polygons."""
    df = gdf.copy()

    # Basic metrics
    df['perimeter'] = df.geometry.length
    df['area'] = df.geometry.area
    df['compactness'] = (4 * math.pi * df['area']) / (df['perimeter'] ** 2)
    df['perimeter_to_area'] = df['perimeter'] / df['area']

    # MRR axis lengths (computed from actual side lengths of the rotated rectangle)
    def mrr_axes(geom):
        mrr = geom.minimum_rotated_rectangle
        coords = list(mrr.exterior.coords)
        side1 = math.hypot(coords[1][0] - coords[0][0], coords[1][1] - coords[0][1])
        side2 = math.hypot(coords[2][0] - coords[1][0], coords[2][1] - coords[1][1])
        return max(side1, side2), min(side1, side2)

    axes = df.geometry.apply(lambda g: pd.Series(mrr_axes(g), index=['major_axis', 'minor_axis']))
    df = pd.concat([df, axes], axis=1)

    # Eccentricity
    def eccentricity(row):
        major, minor = row['major_axis'], row['minor_axis']
        if major == 0 or major == minor:
            return 0.0
        if minor > major:
            major, minor = minor, major
        return math.sqrt(np.clip(1 - (minor ** 2 / major ** 2), 0, 1))

    df['eccentricity'] = df.apply(eccentricity, axis=1)

    return df

FEATURES_5 = ['perimeter', 'area', 'compactness', 'perimeter_to_area', 'eccentricity']

gdf_5feat = extract_5_features(gdf)
print("5-Feature extraction complete:")
gdf_5feat[FEATURES_5].describe()

## 1.4 Option B: 20-Feature Set (Expanded)

The expanded feature set adds shape complexity, radial statistics, and derived metrics. In our evaluation, this set performed equivalently to the 5-feature set -- but is included for completeness and potential use with richer datasets.

In [ ]:
def extract_20_features(gdf):
    """Extract the expanded 20-feature set from a GeoDataFrame of polygons."""
    records = []

    for idx, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty or geom.geom_type != 'Polygon':
            continue

        # --- Basic ---
        area = geom.area
        perimeter = geom.length
        perimeter_to_area = perimeter / area if area > 0 else 0.0

        # --- Shape indices ---
        compactness = (4 * math.pi * area) / (perimeter ** 2) if perimeter > 0 else 0.0
        hull = geom.convex_hull
        hull_area = hull.area
        convexity = area / hull_area if hull_area > 0 else 1.0

        # --- Bounding geometry (MRR) ---
        mrr = geom.minimum_rotated_rectangle
        coords = list(mrr.exterior.coords)
        side1 = math.hypot(coords[1][0] - coords[0][0], coords[1][1] - coords[0][1])
        side2 = math.hypot(coords[2][0] - coords[1][0], coords[2][1] - coords[1][1])
        major = max(side1, side2)
        minor = min(side1, side2)

        # Eccentricity
        if major == 0 or major == minor:
            eccentricity = 0.0
        else:
            eccentricity = math.sqrt(np.clip(1 - (minor ** 2 / major ** 2), 0, 1))

        aspect_ratio = major / minor if minor > 0 else 0.0
        mrr_area = major * minor
        mrr_area_ratio = area / mrr_area if mrr_area > 0 else 0.0

        # --- Complexity ---
        ext_coords = list(geom.exterior.coords)
        n_vertices = len(ext_coords) - 1  # exclude closing vertex
        hull_perimeter = hull.length
        boundary_sinuosity = perimeter / hull_perimeter if hull_perimeter > 0 else 1.0

        # Count concave indentations
        diff = hull.difference(geom)
        if diff.is_empty:
            n_concavities = 0
        elif diff.geom_type == 'Polygon':
            n_concavities = 1
        elif diff.geom_type in ('MultiPolygon', 'GeometryCollection'):
            n_concavities = sum(1 for g in diff.geoms if g.geom_type == 'Polygon' and g.area > 0.01)
        else:
            n_concavities = 0

        # --- Radial stats ---
        centroid = geom.centroid
        distances = np.array([
            math.hypot(c[0] - centroid.x, c[1] - centroid.y)
            for c in ext_coords[:-1]
        ])
        mean_radius = distances.mean() if len(distances) > 0 else 0.0
        radius_std = distances.std() if len(distances) > 0 else 0.0
        radius_cv = radius_std / mean_radius if mean_radius > 0 else 0.0
        radius_ratio = distances.min() / distances.max() if len(distances) > 0 and distances.max() > 0 else 1.0

        # --- Derived ---
        equivalent_diameter = 2 * math.sqrt(area / math.pi) if area > 0 else 0.0
        convex_hull_deficit = hull_area - area

        # L-ratio (from original model)
        if minor == 0 or compactness == 0:
            l_ratio = 0.0
        else:
            l_ratio = (min(major, minor) / max(major, minor)) / (compactness ** 2)

        records.append({
            'area': area, 'perimeter': perimeter, 'perimeter_to_area': perimeter_to_area,
            'compactness': compactness, 'convexity': convexity, 'eccentricity': eccentricity,
            'major_axis': major, 'minor_axis': minor, 'aspect_ratio': aspect_ratio,
            'mrr_area_ratio': mrr_area_ratio, 'n_vertices': n_vertices,
            'boundary_sinuosity': boundary_sinuosity, 'n_concavities': n_concavities,
            'mean_radius': mean_radius, 'radius_std': radius_std, 'radius_cv': radius_cv,
            'radius_ratio': radius_ratio, 'equivalent_diameter': equivalent_diameter,
            'convex_hull_deficit': convex_hull_deficit, 'l_ratio': l_ratio,
        })

    return pd.DataFrame(records, index=gdf.index[:len(records)])

FEATURES_20 = [
    'area', 'perimeter', 'perimeter_to_area', 'compactness', 'convexity',
    'eccentricity', 'major_axis', 'minor_axis', 'aspect_ratio', 'mrr_area_ratio',
    'n_vertices', 'boundary_sinuosity', 'n_concavities', 'mean_radius',
    'radius_std', 'radius_cv', 'radius_ratio', 'equivalent_diameter',
    'convex_hull_deficit', 'l_ratio',
]

features_20 = extract_20_features(gdf)
gdf_20feat = pd.concat([gdf, features_20], axis=1)
print("20-Feature extraction complete:")
gdf_20feat[FEATURES_20].describe()

## 1.5 Exploratory Data Analysis

In [ ]:
TARGET = 'Point_Coun'

# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(gdf[TARGET], bins=25, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Tree Trunk Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Target Distribution')
axes[0].axvline(gdf[TARGET].mean(), color='red', linestyle='--', label=f'Mean={gdf[TARGET].mean():.1f}')
axes[0].axvline(gdf[TARGET].median(), color='orange', linestyle='--', label=f'Median={gdf[TARGET].median():.1f}')
axes[0].legend()

axes[1].hist(np.log1p(gdf[TARGET]), bins=25, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('log(1 + Trunk Count)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log-Transformed Target')
fig.tight_layout()
plt.show()

print(f"\nTarget stats:\n{gdf[TARGET].describe()}")

In [ ]:
import seaborn as sns

# Feature correlations with target (5-feature set)
corr_5 = gdf_5feat[FEATURES_5 + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print("5-Feature correlations with target:")
print(corr_5.to_string())

# Feature-target scatter (5-feature set)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, feat in zip(axes, FEATURES_5):
    ax.scatter(gdf_5feat[feat], gdf_5feat[TARGET], alpha=0.3, s=10)
    ax.set_xlabel(feat)
    ax.set_ylabel('Trunk Count')
    ax.set_title(f'r = {corr_5[feat]:.3f}')
fig.suptitle('5-Feature Set vs Target', fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (20-feature set)
corr_20 = gdf_20feat[FEATURES_20 + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_20, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=ax, annot_kws={'size': 7})
ax.set_title('20-Feature Correlation Matrix (including target)')
fig.tight_layout()
plt.show()

# Target correlations
print("\n20-Feature correlations with target:")
print(corr_20[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).to_string())

## 1.6 Save Feature-Enriched Data

Save the GeoDataFrame with extracted features for use in Notebook 2 (model training). Both feature sets are saved as separate shapefiles.

In [ ]:
# Save 5-feature version
out_5 = gdf_5feat[FEATURES_5 + [TARGET, 'geometry']].copy()
out_5 = gpd.GeoDataFrame(out_5, geometry='geometry', crs=gdf.crs)
out_5.to_file("train_features_5.shp")
print(f"Saved train_features_5.shp ({len(out_5)} rows, {len(FEATURES_5)} features)")

# Save 20-feature version
# Shapefile column names are limited to 10 characters -- use abbreviations
col_map_20 = {
    'area': 'area', 'perimeter': 'perimeter', 'perimeter_to_area': 'p_to_a',
    'compactness': 'compact', 'convexity': 'convexity', 'eccentricity': 'eccentric',
    'major_axis': 'major_ax', 'minor_axis': 'minor_ax', 'aspect_ratio': 'asp_ratio',
    'mrr_area_ratio': 'mrr_a_rat', 'n_vertices': 'n_verts',
    'boundary_sinuosity': 'bnd_sinus', 'n_concavities': 'n_concav',
    'mean_radius': 'mean_rad', 'radius_std': 'rad_std', 'radius_cv': 'rad_cv',
    'radius_ratio': 'rad_ratio', 'equivalent_diameter': 'eq_diam',
    'convex_hull_deficit': 'hull_def', 'l_ratio': 'l_ratio',
}
out_20 = gdf_20feat[FEATURES_20 + [TARGET, 'geometry']].copy()
out_20 = out_20.rename(columns=col_map_20)
out_20 = gpd.GeoDataFrame(out_20, geometry='geometry', crs=gdf.crs)
out_20.to_file("train_features_20.shp")
print(f"Saved train_features_20.shp ({len(out_20)} rows, {len(FEATURES_20)} features)")

# Also save as pickle for Notebook 2 (preserves full column names)
gdf_5feat[FEATURES_5 + [TARGET]].to_pickle("train_features_5.pkl")
gdf_20feat[FEATURES_20 + [TARGET]].to_pickle("train_features_20.pkl")
print("Saved pickle files for Notebook 2")